no GAE, no replay buffer, no parallel workers, no clipping tricks,



1. **Collect one step**  
   Collect a transition tuple from the environment:  

   $
   (s, a, r, s', \text{done})
   $

2. **Critic Update**  
   Minimize the Temporal Difference (TD) error:  

   $
   \mathcal{L}_{\text{critic}} = \left(r + \gamma V(s') - V(s)\right)^2
   $

3. **Compute Advantage**  
   Use the TD error as the advantage estimate:  

   $
   A(s, a) = r + \gamma V(s') - V(s)
   $

  
   *Note: Detach this value when using it in the actor loss to avoid backpropagating through the critic.*

4. **Actor Update**  
   Maximize the expected return by performing gradient ascent on:

   $
   \mathcal{L}_{\text{actor}} = \log \pi(a \mid s) \cdot \text{advantage}
   $






In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical


In [2]:
GAMMA = 0.99
LR_ACTOR = 1e-3
LR_CRITIC = 1e-3
HID = 128
ENV_ID = "CartPole-v1"
EPISODES = 1000
MAX_STEPS = 500               # env terminates earlier anyway
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, HID), nn.ReLU(),
            nn.Linear(HID, a_dim), nn.Softmax(dim=-1)
        )

    def forward(self, s):
        return self.net(s)

class Critic(nn.Module):
    def __init__(self, s_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, HID), nn.ReLU(),
            nn.Linear(HID, 1)      # scalar V(s)
        )

    def forward(self, s):
        return self.net(s)


In [4]:
# ---------- Setup ----------
env = gym.make(ENV_ID)
s_dim = env.observation_space.shape[0]
a_dim = env.action_space.n

actor  = Actor(s_dim, a_dim).to(device)
critic = Critic(s_dim).to(device)

optim_actor  = torch.optim.Adam(actor.parameters(),  lr=LR_ACTOR)
optim_critic = torch.optim.Adam(critic.parameters(), lr=LR_CRITIC)


In [ ]:
# ---------- Training Loop ----------
for ep in range(EPISODES):
    state, _ = env.reset(seed=None)
    ep_return = 0

    for t in range(MAX_STEPS):
        s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)   # shape [1, s_dim]

        # --- Actor: pick action stochastically
        probs = actor(s)
        dist  = Categorical(probs)
        action = dist.sample()                      # tensor scalar
        logp   = dist.log_prob(action)              # will feed into gradient
        a = action.item()

        # --- Step environment
        next_state, reward, done, trunc, _ = env.step(a)
        terminated = done or trunc
        ep_return += reward

        # --- Critic targets
        s_val   = critic(s)                         # V(s)
        s_next  = torch.tensor(next_state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            v_next = critic(s_next) * (1.0 - float(terminated))
            td_target = torch.tensor(reward, device=device).unsqueeze(0) + GAMMA * v_next

        advantage = td_target - s_val               # one‑step TD error

        # --- Update Critic (mean‑squared TD error)
        critic_loss = advantage.pow(2).mean()
        optim_critic.zero_grad()
        critic_loss.backward()
        optim_critic.step()

        # --- Update Actor (policy gradient)
        actor_loss = -(logp * advantage.detach())   # negate -> gradient *ascent*
        optim_actor.zero_grad()
        actor_loss.backward()
        optim_actor.step()

        # --- Move to next state
        state = next_state
        if terminated:
            break
    if ep%50 ==0:
        print(f"Episode {ep:4d} | Return = {ep_return:5.1f}")

Episode    0 | Return =  22.0
Episode   50 | Return =   9.0
Episode  100 | Return =  10.0
Episode  150 | Return =  11.0
Episode  200 | Return =  49.0
Episode  250 | Return =  52.0
Episode  300 | Return =  96.0
Episode  350 | Return =  16.0
